# 9 WorkFlow Analista Senior

### 9.5 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
enriquecer
<br>El Analista Sr corre sus scripts en máquinas virtuales de al menos 128 GB de RAM en Toronto, creando una virtual machine para cada corrida.
<br>Estas virtual machines se auto-suicidarán a los 30 minutos de haber terminado de procesar.

## 9.7  Workflow

## Inicializacion

#### Limpio el ambiente de R

In [ ]:
# Limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

#### Importo librerias

In [ ]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

#### Parámetros (ajustar!)

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 701479

PARAM$experimento <- 9500205
PARAM$dataset <- "analistasr_competencia_2025.csv.gz"

#### Carpeta del Experimento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd(paste0("/content/buckets/b1/exp/", experimento_folder))

#### Logging

In [ ]:
# Comando para registrar la salida del script en un archivo
Log <- function(texto) {
  linea <- paste0(format(Sys.time(), "%Y-%m-%d %H:%M:%S"), " | ", texto, "\n")
  cat(linea, file = "workflow_log.txt", append = TRUE)
}
Log("Inicio del script")

### 9.7.1   Preprocesamiento del dataset

#### 9.7.1.1  DT incorporar dataset

In [ ]:
# Lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 9.7.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [ ]:
if(!require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

In [ ]:
Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice!
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}

In [ ]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [ ]:
AsignarNA_campomeses <- function(pcampo, pmeses) {
  if (pcampo %in% colnames(dataset)) {
    dataset[foto_mes %in% pmeses, paste0(pcampo) := NA]
  }
}

In [ ]:
Corregir_atributo <- function(pcampo, pmeses, pmetodo) {
  # si el campo no existe en el dataset, afuera!
  if (!(pcampo %in% colnames(dataset)))
    return( 1 )

  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return(0)
}

In [ ]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  Log( "Inicio Corregir_Rotas()")

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  Log("Fin Corregir_rotas()")
}


In [ ]:
setorder(dataset, numero_de_cliente, foto_mes)

# PARAM$CA$metodo= "MachineLearning"
PARAM$CA$metodo = "EstadisticaClasica"

if (PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE"))
  Corregir_Rotas(dataset, PARAM$CA$metodo)

#### 9.7.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [ ]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [ ]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.780952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [ ]:
tb_indices <- as.data.table(
  list(
    "IPC" = vIPC,
    "dolar_blue" = vdolar_blue,
    "dolar_oficial" = vdolar_oficial,
    "UVA" = vUVA
  )
)

tb_indices[['foto_mes']] <- vfoto_mes

tb_indices

In [ ]:
drift_UVA <- function(campos_monetarios) {
  Log( "inicio drift_UVA()")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  Log( "fin drift_UVA()")
}

In [ ]:
drift_dolar_oficial <- function(campos_monetarios) {
  Log( "Inicio drift_dolar_oficial()")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  Log( "Fin drift_dolar_oficial()")
}

In [ ]:
drift_dolar_blue <- function(campos_monetarios) {
  Log( "Inicio drift_dolar_blue()")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  Log( "Fin drift_dolar_blue()")
}


In [ ]:
drift_deflacion <- function(campos_monetarios) {
  Log( "Inicio drift_deflacion()")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  Log( "Fin drift_deflacion()")
}


In [ ]:
drift_rank_simple <- function(campos_drift) {
  Log("Inicio drift_rank_simple()")
  for (campo in campos_drift) {
    Log(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  Log("Fin drift_rank_simple()")
}

In [ ]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {
  Log("Inicio drift_rank_cero_fijo()")
  for (campo in campos_drift) {
    Log(paste0(campo, " "))
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  Log( "fin drift_rank_cero_fijo()")
}


In [ ]:
drift_estandarizar <- function(campos_drift) {
  Log("Inicio drift_estandarizar()")
  for (campo in campos_drift) {
    Log(paste0(campo, " "))
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(campo, na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  Log( "Fin drift_estandarizar()")
}


In [ ]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

In [ ]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )

# PARAM$DR$metodo <- "deflacion"
PARAM$DR$metodo <- "UVA"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolarblue(campos_monetarios),
  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


In [ ]:
colnames(dataset)

#### 9.7.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [ ]:
if( !require("lubridate")) install.packages("lubridate", repos = "http://cran.us.r-project.org")
require("lubridate")

In [ ]:
# Esta funcion "atributos presentes" existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos ) {
  atributos <- unique(patributos )
  comun <- intersect(atributos, colnames(dataset))

  return(length(atributos) == length(comun))
}

In [ ]:
AgregarVariables_IntraMes <- function(dataset) {
  Log("Inicio AgregarVariables_IntraMes()")
  gc(verbose= FALSE)

  if( atributos_presentes( c("foto_mes") ))
    dataset[, kmes := foto_mes %% 100]

  # creo un ctr_quarter que tenga en cuenta cuando
  # los clientes hace 3 menos meses que estan
  # ya que seria injusto considerar las transacciones medidas en menor tiempo
  if( atributos_presentes( c("ctrx_quarter") ))
    dataset[, ctrx_quarter_normalizado := as.numeric(ctrx_quarter) ]

  if( atributos_presentes( c("ctrx_quarter", "cliente_antiguedad") ))
    dataset[cliente_antiguedad == 1, ctrx_quarter_normalizado := ctrx_quarter * 5]

  if( atributos_presentes( c("ctrx_quarter", "cliente_antiguedad") ))
    dataset[cliente_antiguedad == 2, ctrx_quarter_normalizado := ctrx_quarter * 2]

  if( atributos_presentes( c("ctrx_quarter", "cliente_antiguedad") ))
    dataset[
      cliente_antiguedad == 3,
      ctrx_quarter_normalizado := ctrx_quarter * 1.2
    ]

   if(atributos_presentes(c("foto_mes")))
    dataset[,foto_mes_formato_fecha := as.Date(paste(substr(dataset$foto_mes,1,4),substr(dataset$foto_mes,5,6),"01",sep='-'))]

  #dataset$foto_mes_formato_fecha <<- as.Date(paste(substr(dataset$foto_mes,1,4),substr(dataset$foto_mes,5,6),"01",sep='-'))

  if(atributos_presentes(c("cantidad_total_transacciones"))){
   auxiliarmenos1 <- dataset[,list(numero_de_cliente,foto_mes_formato_fecha, cantidad_total_transacciones)]
   auxiliarmenos2 <- dataset[,list(numero_de_cliente,foto_mes_formato_fecha,cantidad_total_transacciones)]
   # auxiliarmenos1$foto_mes_formato_fecha <- as.Date(auxiliarmenos1$foto_mes_formato_fecha)
   # auxiliarmenos2$foto_mes_formato_fecha <- as.Date(auxiliarmenos2$foto_mes_formato_fecha)
   auxiliarmenos1$foto_mes_formato_fecha <- auxiliarmenos1$foto_mes_formato_fecha  %m-%  months(1)
   auxiliarmenos2$foto_mes_formato_fecha <- auxiliarmenos2$foto_mes_formato_fecha %m-% months(2)
   auxiliarmenos1$codigo <- paste(auxiliarmenos1$numero_de_cliente,auxiliarmenos1$foto_mes_formato_fecha,sep='-')
   auxiliarmenos2$codigo <- paste(auxiliarmenos2$numero_de_cliente,auxiliarmenos2$foto_mes_formato_fecha,sep='-')

   dataset[, codigo := paste(numero_de_cliente, foto_mes_formato_fecha, sep='-') ]

   dataset[ auxiliarmenos1,
            on = "codigo",
            transaccionesmenos1 := i.cantidad_total_transacciones ]

   dataset[ auxiliarmenos2,
            on = "codigo",
            transaccionesmenos2 := i.cantidad_total_transacciones ]

   dataset[, cantidad_total_transacciones_quarter := rowSums(cbind(cantidad_total_transacciones +
    transaccionesmenos1 + transaccionesmenos2),na.rm=T) ]

   dataset[, codigo := NULL ]
   dataset[, transaccionesmenos1 := NULL ]
   dataset[, transaccionesmenos2 := NULL ]
   dataset[, foto_mes_formato_fecha := NULL ]
   rm(auxiliarmenos1)
   rm(auxiliarmenos2)
  }

  if( atributos_presentes( c("cantidad_total_transacciones_quarter") ))
    dataset[, cantidad_total_transacciones_quarter_normalizado := cantidad_total_transacciones_quarter]

  if( atributos_presentes( c("cantidad_total_transacciones_quarter", "cliente_antiguedad") ))
    dataset[cliente_antiguedad == 1, cantidad_total_transacciones_quarter_normalizado := cantidad_total_transacciones_quarter * 5]

  if( atributos_presentes( c("cantidad_total_transacciones_quarter", "cliente_antiguedad") ))
    dataset[cliente_antiguedad == 2, cantidad_total_transacciones_quarter_normalizado := cantidad_total_transacciones_quarter * 2]

  if( atributos_presentes( c("cantidad_total_transacciones_quarter", "cliente_antiguedad") ))
    dataset[cliente_antiguedad == 3, cantidad_total_transacciones_quarter_normalizado := cantidad_total_transacciones_quarter * 1.2]

  # variable extraida de una tesis de maestria de Irlanda
  if( atributos_presentes( c("mpayroll", "cliente_edad") ))
    dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

  # se crean los nuevos campos para MasterCard  y Visa,
  #  teniendo en cuenta los NA's
  # varias formas de combinar Visa_status y Master_status
  if( atributos_presentes( c("Master_status", "Visa_status") ))
  {
    dataset[, vm_status01 := pmax(Master_status, Visa_status, na.rm = TRUE)]
    dataset[, vm_status02 := Master_status + Visa_status]

    dataset[, vm_status03 := pmax(
      ifelse(is.na(Master_status), 10, Master_status),
      ifelse(is.na(Visa_status), 10, Visa_status)
    )]

    dataset[, vm_status04 := ifelse(is.na(Master_status), 10, Master_status)
      + ifelse(is.na(Visa_status), 10, Visa_status)]

    dataset[, vm_status05 := ifelse(is.na(Master_status), 10, Master_status)
      + 100 * ifelse(is.na(Visa_status), 10, Visa_status)]

    dataset[, vm_status06 := ifelse(is.na(Visa_status),
      ifelse(is.na(Master_status), 10, Master_status),
      Visa_status
    )]

    dataset[, mv_status07 := ifelse(is.na(Master_status),
      ifelse(is.na(Visa_status), 10, Visa_status),
      Master_status
    )]
  }

  # Combino MasterCard y Visa
  if( atributos_presentes( c("Master_mfinanciacion_limite", "Visa_mfinanciacion_limite") ))
    dataset[, vm_mfinanciacion_limite := rowSums(cbind(Master_mfinanciacion_limite, Visa_mfinanciacion_limite), na.rm = TRUE)]

  if( atributos_presentes( c("Master_Fvencimiento", "Visa_Fvencimiento") ))
    dataset[, vm_Fvencimiento := pmin(Master_Fvencimiento, Visa_Fvencimiento, na.rm = TRUE)]

  if( atributos_presentes( c("Master_Finiciomora", "Visa_Finiciomora") ))
    dataset[, vm_Finiciomora := pmin(Master_Finiciomora, Visa_Finiciomora, na.rm = TRUE)]

  if( atributos_presentes( c("Master_msaldototal", "Visa_msaldototal") ))
    dataset[, vm_msaldototal := rowSums(cbind(Master_msaldototal, Visa_msaldototal), na.rm = TRUE)]

  if( atributos_presentes( c("Master_msaldopesos", "Visa_msaldopesos") ))
    dataset[, vm_msaldopesos := rowSums(cbind(Master_msaldopesos, Visa_msaldopesos), na.rm = TRUE)]

  if( atributos_presentes( c("Master_msaldodolares", "Visa_msaldodolares") ))
    dataset[, vm_msaldodolares := rowSums(cbind(Master_msaldodolares, Visa_msaldodolares), na.rm = TRUE)]

  if( atributos_presentes( c("Master_mconsumospesos", "Visa_mconsumospesos") ))
    dataset[, vm_mconsumospesos := rowSums(cbind(Master_mconsumospesos, Visa_mconsumospesos), na.rm = TRUE)]

  if( atributos_presentes( c("Master_mconsumosdolares", "Visa_mconsumosdolares") ))
    dataset[, vm_mconsumosdolares := rowSums(cbind(Master_mconsumosdolares, Visa_mconsumosdolares), na.rm = TRUE)]

  if( atributos_presentes( c("Master_mlimitecompra", "Visa_mlimitecompra") ))
    dataset[, vm_mlimitecompra := rowSums(cbind(Master_mlimitecompra, Visa_mlimitecompra), na.rm = TRUE)]

  if( atributos_presentes( c("Master_madelantopesos", "Visa_madelantopesos") ))
    dataset[, vm_madelantopesos := rowSums(cbind(Master_madelantopesos, Visa_madelantopesos), na.rm = TRUE)]

  if( atributos_presentes( c("Master_madelantodolares", "Visa_madelantodolares") ))
    dataset[, vm_madelantodolares := rowSums(cbind(Master_madelantodolares, Visa_madelantodolares), na.rm = TRUE)]

  if( atributos_presentes( c("Master_fultimo_cierre", "Visa_fultimo_cierre") ))
    dataset[, vm_fultimo_cierre := pmax(Master_fultimo_cierre, Visa_fultimo_cierre, na.rm = TRUE)]

  if( atributos_presentes( c("Master_mpagado", "Visa_mpagado") ))
    dataset[, vm_mpagado := rowSums(cbind(Master_mpagado, Visa_mpagado), na.rm = TRUE)]

  if( atributos_presentes( c("Master_mpagospesos", "Visa_mpagospesos") ))
    dataset[, vm_mpagospesos := rowSums(cbind(Master_mpagospesos, Visa_mpagospesos), na.rm = TRUE)]

  if( atributos_presentes( c("Master_mpagosdolares", "Visa_mpagosdolares") ))
    dataset[, vm_mpagosdolares := rowSums(cbind(Master_mpagosdolares, Visa_mpagosdolares), na.rm = TRUE)]

  if( atributos_presentes( c("Master_fechaalta", "Visa_fechaalta") ))
    dataset[, vm_fechaalta := pmax(Master_fechaalta, Visa_fechaalta, na.rm = TRUE)]

  if( atributos_presentes( c("Master_mconsumototal", "Visa_mconsumototal") ))
    dataset[, vm_mconsumototal := rowSums(cbind(Master_mconsumototal, Visa_mconsumototal), na.rm = TRUE)]

  if( atributos_presentes( c("Master_cconsumos", "Visa_cconsumos") ))
    dataset[, vm_cconsumos := rowSums(cbind(Master_cconsumos, Visa_cconsumos), na.rm = TRUE)]

  if( atributos_presentes( c("Master_cadelantosefectivo", "Visa_cadelantosefectivo") ))
    dataset[, vm_cadelantosefectivo := rowSums(cbind(Master_cadelantosefectivo, Visa_cadelantosefectivo), na.rm = TRUE)]

  if( atributos_presentes( c("Master_mpagominimo", "Visa_mpagominimo") ))
    dataset[, vm_mpagominimo := rowSums(cbind(Master_mpagominimo, Visa_mpagominimo), na.rm = TRUE)]

  # A partir de aqui juego con la suma de Mastercard y Visa
  if( atributos_presentes( c("Master_mlimitecompra", "vm_mlimitecompra") ))
    dataset[, vmr_Master_mlimitecompra := Master_mlimitecompra / vm_mlimitecompra]

  if( atributos_presentes( c("Visa_mlimitecompra", "vm_mlimitecompra") ))
    dataset[, vmr_Visa_mlimitecompra := Visa_mlimitecompra / vm_mlimitecompra]

  if( atributos_presentes( c("vm_msaldototal", "vm_mlimitecompra") ))
    dataset[, vmr_msaldototal := vm_msaldototal / vm_mlimitecompra]

  if( atributos_presentes( c("vm_msaldopesos", "vm_mlimitecompra") ))
    dataset[, vmr_msaldopesos := vm_msaldopesos / vm_mlimitecompra]

  if( atributos_presentes( c("vm_msaldopesos", "vm_msaldototal") ))
    dataset[, vmr_msaldopesos2 := vm_msaldopesos / vm_msaldototal]

  if( atributos_presentes( c("vm_msaldodolares", "vm_mlimitecompra") ))
    dataset[, vmr_msaldodolares := vm_msaldodolares / vm_mlimitecompra]

  if( atributos_presentes( c("vm_msaldodolares", "vm_msaldototal") ))
    dataset[, vmr_msaldodolares2 := vm_msaldodolares / vm_msaldototal]

  if( atributos_presentes( c("vm_mconsumospesos", "vm_mlimitecompra") ))
    dataset[, vmr_mconsumospesos := vm_mconsumospesos / vm_mlimitecompra]

  if( atributos_presentes( c("vm_mconsumosdolares", "vm_mlimitecompra") ))
    dataset[, vmr_mconsumosdolares := vm_mconsumosdolares / vm_mlimitecompra]

  if( atributos_presentes( c("vm_madelantopesos", "vm_mlimitecompra") ))
    dataset[, vmr_madelantopesos := vm_madelantopesos / vm_mlimitecompra]

  if( atributos_presentes( c("vm_madelantodolares", "vm_mlimitecompra") ))
    dataset[, vmr_madelantodolares := vm_madelantodolares / vm_mlimitecompra]

  if( atributos_presentes( c("vm_mpagado", "vm_mlimitecompra") ))
    dataset[, vmr_mpagado := vm_mpagado / vm_mlimitecompra]

  if( atributos_presentes( c("vm_mpagospesos", "vm_mlimitecompra") ))
    dataset[, vmr_mpagospesos := vm_mpagospesos / vm_mlimitecompra]

  if( atributos_presentes( c("vm_mpagosdolares", "vm_mlimitecompra") ))
    dataset[, vmr_mpagosdolares := vm_mpagosdolares / vm_mlimitecompra]

  if( atributos_presentes( c("vm_mconsumototal", "vm_mlimitecompra") ))
    dataset[, vmr_mconsumototal := vm_mconsumototal / vm_mlimitecompra]

  if( atributos_presentes( c("vm_mpagominimo", "vm_mlimitecompra") ))
    dataset[, vmr_mpagominimo := vm_mpagominimo / vm_mlimitecompra]

  Log("Fin de la seccion donde se hacen cambios con variables nuevas dentro del script base")

  # --------------------------------------------------------------------------
  # INICIO de la seccion donde se agregan variables nuevas manuales
  # --------------------------------------------------------------------------
  Log("-- Inicio agregado variables nuevas manuales --")

  # --- helpers cortos ---
  as_date_ym <- function(x) as.Date(paste0(substr(x,1,4), "-", substr(x,5,6), "-01"))

  # año/mes (estacionalidad simple)
  if (atributos_presentes(c("foto_mes"))) {
    dataset[, kmes  := as.integer(foto_mes %% 100)]
    dataset[, kanio := as.integer(foto_mes %/% 100)]
    dataset[, foto_mes_date := as_date_ym(foto_mes)]
  }

  # buckets de antigüedad (no lineales)
  if (atributos_presentes(c("cliente_antiguedad"))) {
    dataset[, antig_bucket := fifelse(cliente_antiguedad < 3, "nuevo",
                              fifelse(cliente_antiguedad < 12, "0-12",
                              fifelse(cliente_antiguedad < 24, "12-24",
                              fifelse(cliente_antiguedad < 60, "24-60", "60+"))))]
  }

  # -------- UTILIZACIONES Y PROPORCIONES (por tarjeta y combinado) --------
  # Utilización (saldo / límite) y proporciones clave
  if (atributos_presentes(c("Master_msaldototal","Master_mlimitecompra"))) {
    dataset[, master_util := Master_msaldototal / pmax(Master_mlimitecompra, 1)]
  }
  if (atributos_presentes(c("Visa_msaldototal","Visa_mlimitecompra"))) {
    dataset[, visa_util := Visa_msaldototal / pmax(Visa_mlimitecompra, 1)]
  }
  if (atributos_presentes(c("vm_msaldototal","vm_mlimitecompra"))) {
    dataset[, util_total := vm_msaldototal / pmax(vm_mlimitecompra, 1)]
  }

  # Proporción de adelantos (cash advance) vs consumo total
  if (atributos_presentes(c("vm_madelantopesos","vm_madelantodolares","vm_mconsumototal"))) {
    dataset[, cash_adv_share := (vm_madelantopesos + vm_madelantodolares) / pmax(vm_mconsumototal, 1)]
  }

  # Proporción de consumos en dólares (riesgo FX)
  if (atributos_presentes(c("vm_mconsumosdolares","vm_mconsumototal"))) {
    dataset[, fx_consume_share := vm_mconsumosdolares / pmax(vm_mconsumototal, 1)]
  }

  # Pago relativo (pago / saldo) y pago mínimo relativo
  if (atributos_presentes(c("vm_mpagado","vm_msaldototal"))) {
    dataset[, pay_ratio := vm_mpagado / pmax(vm_msaldototal, 1)]
  }
  if (atributos_presentes(c("vm_mpagominimo","vm_mpagado"))) {
    dataset[, paymin_ratio := vm_mpagominimo / pmax(vm_mpagado, 1)]
  }

  # Ingreso vs gasto (capacidad de pago aproximada)
  if (atributos_presentes(c("mpayroll","vm_mconsumototal"))) {
    dataset[, consume_over_payroll := vm_mconsumototal / pmax(mpayroll, 1)]
  }
  if (atributos_presentes(c("mpayroll","vm_msaldototal"))) {
    dataset[, debt_over_payroll := vm_msaldototal / pmax(mpayroll, 1)]
  }

  # -------- MOROSIDAD / RECENCIA --------
  # Flags y recencia de mora (en meses)
  if (atributos_presentes(c("Master_Finiciomora"))) dataset[, master_mora_flag := !is.na(Master_Finiciomora)]
  if (atributos_presentes(c("Visa_Finiciomora")))   dataset[, visa_mora_flag   := !is.na(Visa_Finiciomora)]
  if (atributos_presentes(c("Master_Finiciomora","Visa_Finiciomora")))
    dataset[, any_mora_flag := master_mora_flag | visa_mora_flag]

  # recencia_mora: meses desde inicio de mora (si hay fecha; requiere foto_mes_date)
  if (atributos_presentes(c("foto_mes")) && atributos_presentes(c("Master_Finiciomora","Visa_Finiciomora"))) {
    # tomo la más reciente entre ambas (si existieran ambas)
    dataset[, finiciomora_date := as.Date(pmax(Master_Finiciomora, Visa_Finiciomora, na.rm=TRUE), origin="1970-01-01")]
    dataset[, recencia_mora_m := fifelse(!is.na(finiciomora_date),
                                        pmax(0, 12*(year(foto_mes_date)-year(finiciomora_date)) + (month(foto_mes_date)-month(finiciomora_date))),
                                        NA_real_)]
  }

  # -------- TENDENCIAS / CAMBIOS (Δ vs mes anterior por cliente) --------
  # Requiere ordenar por cliente y mes; usa shift por grupo
  if (atributos_presentes(c("numero_de_cliente","foto_mes"))) {
    setorder(dataset, numero_de_cliente, foto_mes)
    # Cambios en consumo y saldo total
    if (atributos_presentes(c("vm_mconsumototal"))) {
      dataset[, consumo_t1 := shift(vm_mconsumototal, 1, type="lag"), by = numero_de_cliente]
      dataset[, d1_consume := vm_mconsumototal - consumo_t1]
    }
    if (atributos_presentes(c("vm_msaldototal"))) {
      dataset[, saldo_t1 := shift(vm_msaldototal, 1, type="lag"), by = numero_de_cliente]
      dataset[, d1_saldo := vm_msaldototal - saldo_t1]
    }
    # Intensidad de actividad: conteo de consumos (si existe)
    if (atributos_presentes(c("vm_cconsumos"))) {
      dataset[, ccons_t1 := shift(vm_cconsumos, 1, type="lag"), by = numero_de_cliente]
      dataset[, d1_ccons := vm_cconsumos - ccons_t1]
    }
  }

  # -------- INTERACCIONES BARATAS (señales no lineales simples) --------
  # Utilización * cash-advance share (riesgo revolvente + cash)
  if (atributos_presentes(c("util_total","cash_adv_share"))) {
    dataset[, util_x_cash := util_total * cash_adv_share]
  }

  # Pago relativo versus utilización (capacidad de bajar deuda)
  if (atributos_presentes(c("pay_ratio","util_total"))) {
    dataset[, pay_vs_util := pay_ratio - util_total]
  }

  # Consumo relativo vs límite (proxy uso del límite)
  if (atributos_presentes(c("vm_mconsumototal","vm_mlimitecompra"))) {
    dataset[, consume_over_limit := vm_mconsumototal / pmax(vm_mlimitecompra, 1)]
  }

  # Limpieza auxiliar
  dataset[, c("foto_mes_date","finiciomora_date") := NULL]

  # Orden temporal por cliente (indispensable para lag/ventanas)
  if (atributos_presentes(c("numero_de_cliente","foto_mes"))) {
    setorder(dataset, numero_de_cliente, foto_mes)
  }

  # ---- 1) Lags y tendencias recientes (sin mirar futuro) ----
  # Transacciones totales (elige la que tengas: 'cantidad_total_transacciones' o tu quarter)
  if (atributos_presentes(c("numero_de_cliente","cantidad_total_transacciones"))) {
    dataset[, trx_t1 := shift(cantidad_total_transacciones, 1L), by = numero_de_cliente]
    dataset[, d1_trx := cantidad_total_transacciones - trx_t1]
    dataset[, pd1_trx := fifelse(trx_t1 > 0, d1_trx / trx_t1, NA_real_)]
    
    # promedio móvil de los últimos 3 meses (hasta t-1) para comparar el valor actual
    dataset[, trx_t1_ma3 := frollmean(shift(cantidad_total_transacciones, 1L), 3L, na.rm = TRUE), 
            by = numero_de_cliente]
    dataset[, gap_trx_vs_ma3 := cantidad_total_transacciones - trx_t1_ma3]
  }

  # Consumo total
  if (atributos_presentes(c("numero_de_cliente","vm_mconsumototal"))) {
    dataset[, cons_t1 := shift(vm_mconsumototal, 1L), by = numero_de_cliente]
    dataset[, d1_cons := vm_mconsumototal - cons_t1]
    dataset[, pd1_cons := fifelse(cons_t1 > 0, d1_cons / cons_t1, NA_real_)]
    dataset[, cons_t1_ma3 := frollmean(shift(vm_mconsumototal, 1L), 3L, na.rm = TRUE),
            by = numero_de_cliente]
    dataset[, gap_cons_vs_ma3 := vm_mconsumototal - cons_t1_ma3]
  }

  # Saldo total
  if (atributos_presentes(c("numero_de_cliente","vm_msaldototal"))) {
    dataset[, saldo_t1 := shift(vm_msaldototal, 1L), by = numero_de_cliente]
    dataset[, d1_saldo := vm_msaldototal - saldo_t1]
    dataset[, pd1_saldo := fifelse(saldo_t1 > 0, d1_saldo / saldo_t1, NA_real_)]
  }

  # Payroll (ingreso)
  if (atributos_presentes(c("numero_de_cliente","mpayroll"))) {
    dataset[, pay_t1 := shift(mpayroll, 1L), by = numero_de_cliente]
    dataset[, pd1_pay := fifelse(pay_t1 > 0, (mpayroll - pay_t1) / pay_t1, NA_real_)]
  }

  # ---- 2) Racha de inactividad (meses seguidos con 0 transacciones) ----
  if (atributos_presentes(c("numero_de_cliente","cantidad_total_transacciones"))) {
    dataset[, active_flag := fifelse(cantidad_total_transacciones > 0, 1L, 0L)]
    # grupos delimitados por meses con actividad (=1); contamos racha de ceros por grupo
    dataset[, grp_act := cumsum(active_flag == 1L), by = numero_de_cliente]
    dataset[active_flag == 0L, inactivity_streak := seq_len(.N), by = .(numero_de_cliente, grp_act)]
    dataset[active_flag == 1L, inactivity_streak := 0L]
    dataset[, c("grp_act") := NULL]
  }

  # ---- 3) Cambios de límite y uso del límite ----
  if (atributos_presentes(c("numero_de_cliente","vm_mlimitecompra"))) {
    dataset[, lim_t1 := shift(vm_mlimitecompra, 1L), by = numero_de_cliente]
    dataset[, d1_limit := vm_mlimitecompra - lim_t1]
    dataset[, pd1_limit := fifelse(lim_t1 > 0, d1_limit / lim_t1, NA_real_)]
  }
  if (atributos_presentes(c("vm_mconsumototal","vm_mlimitecompra"))) {
    dataset[, consume_over_limit := vm_mconsumototal / pmax(vm_mlimitecompra, 1)]
  }

  # ---- 4) Engagement barato (0–100 aprox) ----
  # Combina actividad y uso (ajustá pesos si querés)
  if (atributos_presentes(c("cantidad_total_transacciones","vm_mconsumototal","vm_mlimitecompra"))) {
    dataset[, eng_score := 
      scales::rescale(cantidad_total_transacciones, to = c(0,60), from = range(cantidad_total_transacciones, na.rm = TRUE)) +
      scales::rescale(consume_over_limit,        to = c(0,40), from = range(consume_over_limit,        na.rm = TRUE))
    ]
  }

  # ---- 5) Cambios de estado de tarjetas (proxy de cierres/bloqueos) ----
  if (atributos_presentes(c("numero_de_cliente","Master_status"))) {
    dataset[, master_status_t1 := shift(Master_status, 1L), by = numero_de_cliente]
    dataset[, master_status_chg := !(is.na(Master_status) & is.na(master_status_t1)) &
                                    (Master_status != master_status_t1)]
  }
  if (atributos_presentes(c("numero_de_cliente","Visa_status"))) {
    dataset[, visa_status_t1 := shift(Visa_status, 1L), by = numero_de_cliente]
    dataset[, visa_status_chg := !(is.na(Visa_status) & is.na(visa_status_t1)) &
                                  (Visa_status != visa_status_t1)]
  }
  if (atributos_presentes(c("visa_status_chg","master_status_chg"))) {
    dataset[, any_card_status_chg := fifelse(isTRUE(visa_status_chg) | isTRUE(master_status_chg), 1L, 0L)]
  }

  # --------------------------------------------------------------------------
  # FIN de la seccion donde se agregan variables nuevas manuales
  # --------------------------------------------------------------------------
  Log("-- Fin agregado variables nuevas manuales --")

  # Valvula de seguridad para evitar valores infinitos
  # paso los infinitos a NULOS
  infinitos <- lapply(
    names(dataset),
    function(.name) dataset[, sum(is.infinite(get(.name)))]
  )

  infinitos_qty <- sum(unlist(infinitos))
  if (infinitos_qty > 0) {
    Log(
      paste0(
        "ATENCION, hay", infinitos_qty,
        "valores infinitos en tu dataset. Seran pasados a NA"
      )
    )
    dataset[mapply(is.infinite, dataset)] <<- NA
  }


  # Valvula de seguridad para evitar valores NaN  que es 0/0
  # paso los NaN a 0 , decision polemica si las hay
  # se invita a asignar un valor razonable segun la semantica del campo creado
  nans <- lapply(
    names(dataset),
    function(.name) dataset[, sum(is.nan(get(.name)))]
  )

  nans_qty <- sum(unlist(nans))
  if (nans_qty > 0) {
    Log(
      paste0(
        "ATENCION, hay", nans_qty,
        "valores NaN 0/0 en tu dataset. Seran pasados arbitrariamente a NA"
      )
    )

    dataset[mapply(is.nan, dataset)] <<- NA
  }

  Log("Fin AgregarVariables_IntraMes()")
}


In [ ]:
AgregarVariables_IntraMes(dataset)

In [ ]:
# Visualizo las columas del dataset a esta etapa
ncol(dataset)
colnames(dataset)

#### 9.7.1.4  FEhist Feature Engineering historico

El Feature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [ ]:
if (!require("Rcpp")) install.packages("Rcpp", repos = "http://cran.us.r-project.org")
require("Rcpp")

In [ ]:
# Se calculan para los 6 meses previos el minimo, maximo y tendencia calculada con cuadrados minimos
# la formula de calculo de la tendencia puede verse en
#  https://stats.libretexts.org/Bookshelves/Introductory_Statistics/Book%3A_Introductory_Statistics_(Shafer_and_Zhang)/10%3A_Correlation_and_Regression/10.04%3A_The_Least_Squares_Regression_Line
# para la máxima velocidad esta funcion esta escrita en lenguaje C, y no en la porqueria de R o Python

cppFunction("NumericVector fhistC(NumericVector pcolumna, IntegerVector pdesde )
{
  /* Aqui se cargan los valores para la regresion */
  double  x[100] ;
  double  y[100] ;

  int n = pcolumna.size();
  NumericVector out( 5*n );

  for(int i = 0; i < n; i++)
  {
    //lag
    if( pdesde[i]-1 < i )  out[ i + 4*n ]  =  pcolumna[i-1] ;
    else                   out[ i + 4*n ]  =  NA_REAL ;


    int  libre    = 0 ;
    int  xvalor   = 1 ;

    for( int j= pdesde[i]-1;  j<=i; j++ )
    {
       double a = pcolumna[j] ;

       if( !R_IsNA( a ) )
       {
          y[ libre ]= a ;
          x[ libre ]= xvalor ;
          libre++ ;
       }

       xvalor++ ;
    }

    /* Si hay al menos dos valores */
    if( libre > 1 )
    {
      double  xsum  = x[0] ;
      double  ysum  = y[0] ;
      double  xysum = xsum * ysum ;
      double  xxsum = xsum * xsum ;
      double  vmin  = y[0] ;
      double  vmax  = y[0] ;

      for( int h=1; h<libre; h++)
      {
        xsum  += x[h] ;
        ysum  += y[h] ;
        xysum += x[h]*y[h] ;
        xxsum += x[h]*x[h] ;

        if( y[h] < vmin )  vmin = y[h] ;
        if( y[h] > vmax )  vmax = y[h] ;
      }

      out[ i ]  =  (libre*xysum - xsum*ysum)/(libre*xxsum -xsum*xsum) ;
      out[ i + n ]    =  vmin ;
      out[ i + 2*n ]  =  vmax ;
      out[ i + 3*n ]  =  ysum / libre ;
    }
    else
    {
      out[ i       ]  =  NA_REAL ;
      out[ i + n   ]  =  NA_REAL ;
      out[ i + 2*n ]  =  NA_REAL ;
      out[ i + 3*n ]  =  NA_REAL ;
    }
  }

  return  out;
}")


In [ ]:
# Calcula la tendencia de las variables cols de los ultimos 6 meses
#   La tendencia es la pendiente de la recta que ajusta por cuadrados minimos
#   La funcionalidad de ratioavg es autoria de Daiana Sparta, UAustral 2021

TendenciaYmuchomas <- function(
    dataset, cols, ventana = 6, tendencia = TRUE,
    minimo = TRUE, maximo = TRUE, promedio = TRUE,
    ratioavg = FALSE, ratiomax = FALSE) {
  gc(verbose= FALSE)
  # Esta es la cantidad de meses que utilizo para la historia
  ventana_regresion <- ventana

  last <- nrow(dataset)

  # creo el vector_desde que indica cada ventana
  # de esta forma se acelera el procesamiento ya que lo hago una sola vez
  vector_ids <- dataset[ , numero_de_cliente ]

  vector_desde <- seq(
    -ventana_regresion + 2,
    nrow(dataset) - ventana_regresion + 1
  )

  vector_desde[1:ventana_regresion] <- 1

  for (i in 2:last) {
    if (vector_ids[i - 1] != vector_ids[i]) {
      vector_desde[i] <- i
    }
  }
  for (i in 2:last) {
    if (vector_desde[i] < vector_desde[i - 1]) {
      vector_desde[i] <- vector_desde[i - 1]
    }
  }

  for (campo in cols) {
    nueva_col <- fhistC(dataset[, get(campo)], vector_desde)

    if (tendencia) {
      dataset[, paste0(campo, "_tend", ventana) :=
        nueva_col[(0 * last + 1):(1 * last)]]
    }

    if (minimo) {
      dataset[, paste0(campo, "_min", ventana) :=
        nueva_col[(1 * last + 1):(2 * last)]]
    }

    if (maximo) {
      dataset[, paste0(campo, "_max", ventana) :=
        nueva_col[(2 * last + 1):(3 * last)]]
    }

    if (promedio) {
      dataset[, paste0(campo, "_avg", ventana) :=
        nueva_col[(3 * last + 1):(4 * last)]]
    }

    if (ratioavg) {
      dataset[, paste0(campo, "_ratioavg", ventana) :=
        get(campo) / nueva_col[(3 * last + 1):(4 * last)]]
    }

    if (ratiomax) {
      dataset[, paste0(campo, "_ratiomax", ventana) :=
        get(campo) / nueva_col[(2 * last + 1):(3 * last)]]
    }
  }
}


In [ ]:
# Feature Engineering Historico
setorder(dataset, numero_de_cliente, foto_mes)

# Todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy(
  setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
  )
)
# Solo las numericas
cols_numericas <- names(dataset)[sapply(dataset, is.numeric)]
cols_lagueables <- intersect(cols_lagueables, cols_numericas)

# https://rdrr.io/cran/data.table/man/shift.html

# Lags de orden 1
dataset[,
  paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
  by = numero_de_cliente,
  .SDcols = cols_lagueables
]

# Lags de orden 2
dataset[,
  paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
  by = numero_de_cliente,
  .SDcols = cols_lagueables
]

# Lags de orden 3
dataset[,
  paste0(cols_lagueables, "_lag3") := shift(.SD, 3, NA, "lag"),
  by = numero_de_cliente,
  .SDcols = cols_lagueables
]

# Agrego los delta lags
for (vcol in cols_lagueables) {
  dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
  dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
  dataset[, paste0(vcol, "_delta3") := get(vcol) - get(paste0(vcol, "_lag3"))]
}


In [ ]:
# Parametros de Feature Engineering Historico de Tendencias
PARAM$FE_hist$Tendencias$run <- TRUE
PARAM$FE_hist$Tendencias$ventana <- 6
PARAM$FE_hist$Tendencias$tendencia <- TRUE
PARAM$FE_hist$Tendencias$minimo <- FALSE
PARAM$FE_hist$Tendencias$maximo <- FALSE
PARAM$FE_hist$Tendencias$promedio <- FALSE
PARAM$FE_hist$Tendencias$ratioavg <- FALSE
PARAM$FE_hist$Tendencias$ratiomax <- FALSE


In [ ]:
cols_lagueables <- intersect(cols_lagueables, colnames(dataset))
setorder(dataset, numero_de_cliente, foto_mes)

if (PARAM$FE_hist$Tendencias$run) {
  TendenciaYmuchomas(dataset,
    cols = cols_lagueables,
    ventana = PARAM$FE_hist$Tendencias$ventana, # 6 meses de historia
    tendencia = PARAM$FE_hist$Tendencias$tendencia,
    minimo = PARAM$FE_hist$Tendencias$minimo,
    maximo = PARAM$FE_hist$Tendencias$maximo,
    promedio = PARAM$FE_hist$Tendencias$promedio,
    ratioavg = PARAM$FE_hist$Tendencias$ratioavg,
    ratiomax = PARAM$FE_hist$Tendencias$ratiomax
  )
}


Verificacion de los campos recien creados

In [ ]:
ncol(dataset)
colnames(dataset)

#### 9.7.1.5  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest



In [ ]:
if (!require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

In [ ]:
AgregaVarRandomForest <- function() {
  Log("Inicio AgregaVarRandomForest()")
  gc(verbose= FALSE)
  dataset[, clase01 := 0L ]
  dataset[clase_ternaria %in% PARAM$FE_rf$train$clase01_valor1, clase01 := 1L]

  campos_buenos <- setdiff(
    colnames(dataset),
    c( "clase_ternaria", "clase01")
  )

  dataset[, entrenamiento :=
    as.integer( foto_mes %in% PARAM$FE_rf$train$training )]

  dtrain <- lgb.Dataset(
    data = data.matrix(dataset[entrenamiento == TRUE, campos_buenos, with = FALSE]),
    label = dataset[entrenamiento == TRUE, clase01],
    free_raw_data = FALSE
  )

  modelo <- lgb.train(
    data = dtrain,
    param = PARAM$FE_rf$lgb_param,
    verbose = -100
  )

  Log("Fin construccion RandomForest")

  Log("Grabo el modelo RF en el archivo  modelo.model")
  lgb.save(modelo, file="modelo.model" )

  qarbolitos <- copy(PARAM$FE_rf$lgb_param$num_iterations)

  periodos <- dataset[ , unique( foto_mes ) ]

  for( periodo in  periodos ) {
    datamatrix <- data.matrix(dataset[ foto_mes== periodo, campos_buenos, with = FALSE])

    Log(paste0("Prediccion, periodo = ", periodo))
    prediccion <- predict(
      modelo,
      datamatrix,
      type = "leaf"
    )

    for( arbolito in 1:qarbolitos ) {
      hojas_arbol <- unique(prediccion[ , arbolito])

      for (pos in 1:length(hojas_arbol)) {
        # el numero de nodo de la hoja, estan salteados
        nodo_id <- hojas_arbol[pos]
        dataset[ foto_mes== periodo, paste0(
          "rf_", sprintf("%03d", arbolito),
            "_", sprintf("%03d", nodo_id)
        ) :=  as.integer( nodo_id == prediccion[ , arbolito]) ]

      }

      rm( hojas_arbol )
    }

    rm( prediccion )
    rm( datamatrix )
    gc(verbose= FALSE)
  }

  gc(verbose= FALSE)

  # borro clase01 , no debe ensuciar el dataset
  dataset[ , clase01 := NULL ]
}

In [ ]:
# Parametros de Feature Engineering  a partir de hojas de Random Forest

# Estos CUATRO parametros son los que se deben modificar
PARAM$FE_rf$arbolitos= 20
PARAM$FE_rf$hojas_por_arbol= 16
PARAM$FE_rf$datos_por_hoja= 1000
PARAM$FE_rf$mtry_ratio= 0.2

# Estos son quasi fijos
PARAM$FE_rf$train$clase01_valor1 <- c("BAJA+2", "BAJA+1")
PARAM$FE_rf$train$training <- c( 202101, 202102, 202103)

# Estos TAMBIEN son quasi fijos
PARAM$FE_rf$lgb_param <-list(
    # parametros que se pueden cambiar
    num_iterations = PARAM$FE_rf$arbolitos,
    num_leaves  = PARAM$FE_rf$hojas_por_arbol,
    min_data_in_leaf = PARAM$FE_rf$datos_por_hoja,
    feature_fraction_bynode  = PARAM$FE_rf$mtry_ratio,

    # para que LightGBM emule Random Forest
    boosting = "rf",
    bagging_fraction = ( 1.0 - 1.0/exp(1.0) ),
    bagging_freq = 1.0,
    feature_fraction = 1.0,

    # genericos de LightGBM
    max_bin = 31L,
    objective = "binary",
    first_metric_only = TRUE,
    boost_from_average = TRUE,
    feature_pre_filter = FALSE,
    force_row_wise = TRUE,
    verbosity = -100,
    max_depth = -1L,
    min_gain_to_split = 0.0,
    min_sum_hessian_in_leaf = 0.001,
    lambda_l1 = 0.0,
    lambda_l2 = 0.0,

    pos_bagging_fraction = 1.0,
    neg_bagging_fraction = 1.0,
    is_unbalance = FALSE,
    scale_pos_weight = 1.0,

    drop_rate = 0.1,
    max_drop = 50,
    skip_drop = 0.5,

    extra_trees = FALSE
  )

In [ ]:
# Feature Engineering agregando variables de Random Forest
AgregaVarRandomForest()

# Obtengo los nombres de las columnas RF
rf_columns <- names(dataset)[names(dataset) %like% "^rf"]

# Agrego variables estadisticas sobre las columnas RF
dataset[, min_rf  := do.call(pmin, c(.SD, na.rm = TRUE)), .SDcols = rf_columns]
dataset[, max_rf  := do.call(pmax, c(.SD, na.rm = TRUE)), .SDcols = rf_columns]
dataset[, sum_rf := rowSums(.SD, na.rm = TRUE), .SDcols = rf_columns]
dataset[, mean_rf := rowMeans(.SD, na.rm = TRUE), .SDcols = rf_columns]
dataset[, sd_rf   := apply(.SD, 1, sd, na.rm = TRUE),     .SDcols = rf_columns]
dataset[, nonzero_rf := rowSums(.SD != 0, na.rm = TRUE),  .SDcols = rf_columns]

In [ ]:
ncol(dataset)
colnames(dataset)

#### 9.7.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Analista Sr*

El objetivo de esta etapa NO es mejorar el modelo predictivo

El objetivo es eliminar campos poco importantes para hacer espacio a nuevos campos, debido a las restricciones de memoria RAM.

In [ ]:
VPOS_CORTE <- c()

fganancia_lgbm_meseta <- function(probs, datos) {
  vlabels <- get_field(datos, "label")
  vpesos <- get_field(datos, "weight")

  tbl <- as.data.table(list(
    "prob" = probs,
    "gan" = ifelse(vlabels == 1 & vpesos > 1, PARAM$CN$train$gan1, PARAM$CN$train$gan0)
  ))

  setorder(tbl, -prob)
  tbl[, posicion := .I]
  tbl[, gan_acum := cumsum(gan)]
  setorder(tbl, -gan_acum) # voy por la meseta

  gan <- mean(tbl[1:500, gan_acum]) # meseta de tamaño 500

  pos_meseta <- tbl[1:500, median(posicion)]
  VPOS_CORTE <<- c(VPOS_CORTE, pos_meseta)

  return(list(
    "name" = "ganancia",
    "value" = gan,
    "higher_better" = TRUE
  ))
}


In [ ]:
# Elimina del dataset las variables que estan por debajo
#  de la capa geologica de canaritos
# se llama varias veces, luego de agregar muchas variables nuevas,
#  para ir reduciendo la cantidad de variables
# y así hacer lugar a nuevas variables importantes

GVEZ <- 1

campitos <- c( "numero_de_cliente", "foto_mes", "clase_ternaria" )

CanaritosAsesinos <- function(
  canaritos_ratio,
  canaritos_desvios,
  canaritos_semilla) {

  Log("Inicio CanaritosAsesinos()")
  gc(verbose= FALSE)
  dataset[, clase01 := 0L ]
  dataset[ clase_ternaria %in% PARAM$CN$train$clase01_valor1,
      clase01 := 1L ]

  set.seed(canaritos_semilla, kind = "L'Ecuyer-CMRG")
  for (i in 1:(ncol(dataset) * canaritos_ratio)) {
    dataset[, paste0("canarito", i) := runif(nrow(dataset))]
  }

  campos_buenos <- setdiff(
    colnames(dataset),
    c( campitos, "clase01")
  )

  azar <- runif(nrow(dataset))

  dataset[, entrenamiento :=
    as.integer( foto_mes %in% PARAM$CN$train$training &
      (clase01 == 1 | azar < PARAM$CN$train$undersampling))]

  dtrain <- lgb.Dataset(
    data = data.matrix(dataset[entrenamiento == TRUE, campos_buenos, with = FALSE]),
    label = dataset[entrenamiento == TRUE, clase01],
    weight = dataset[
      entrenamiento == TRUE,
      ifelse(clase_ternaria %in% PARAM$CN$train$positivos, 1.0000001, 1.0)
    ],
    free_raw_data = FALSE
  )

  dvalid <- lgb.Dataset(
    data = data.matrix(dataset[foto_mes %in% PARAM$CN$train$validation, campos_buenos, with = FALSE]),
    label = dataset[foto_mes %in% PARAM$CN$train$validation, clase01],
    weight = dataset[
      foto_mes %in% PARAM$CN$train$validation,
      ifelse( clase_ternaria %in% PARAM$CN$train$positivos, 1.0000001, 1.0)
    ],
    free_raw_data = FALSE
  )


  param <- list(
    objective = "binary",
    metric = "custom",
    first_metric_only = TRUE,
    boost_from_average = TRUE,
    feature_pre_filter = FALSE,
    verbosity = -100,
    seed = canaritos_semilla,
    max_depth = -1, # -1 significa no limitar,  por ahora lo dejo fijo
    min_gain_to_split = 0.0, # por ahora, lo dejo fijo
    lambda_l1 = 0.0, # por ahora, lo dejo fijo
    lambda_l2 = 0.0, # por ahora, lo dejo fijo
    max_bin = 31, # por ahora, lo dejo fijo
    num_iterations = 9999, # un numero grande, lo limita early_stopping_rounds
    force_row_wise = TRUE, # para que los alumnos no se atemoricen con  warning
    learning_rate = 0.065,
    feature_fraction = 1.0, # lo seteo en 1
    min_data_in_leaf = 260,
    num_leaves = 60,
    early_stopping_rounds = 200,
    num_threads = 1
  )

  set.seed(canaritos_semilla, kind = "L'Ecuyer-CMRG")
  modelo <- lgb.train(
    data = dtrain,
    valids = list(valid = dvalid),
    eval = fganancia_lgbm_meseta,
    param = param,
    verbose = -100
  )

  tb_importancia <- lgb.importance(model = modelo)
  tb_importancia[, pos := .I]

  fwrite(tb_importancia,
    file = paste0("impo_", GVEZ, ".txt"),
    sep = "\t"
  )

  GVEZ <<- GVEZ + 1

  umbral <- tb_importancia[
    Feature %like% "canarito",
    median(pos) + canaritos_desvios * sd(pos)
  ] # Atencion corto en la mediana mas desvios!!

  col_utiles <- tb_importancia[
    pos < umbral & !(Feature %like% "canarito"),
    Feature
  ]

  col_utiles <- unique(c(
    col_utiles,
    c(campitos, "mes")
  ))

  col_inutiles <- setdiff(colnames(dataset), col_utiles)

  dataset[, (col_inutiles) := NULL]

  Log("Fin CanaritosAsesinos()")

  return( tb_importancia )
}


In [ ]:
# Estos DOS parametros son los que se deben modificar
PARAM$CN$ratio <- 0.2
PARAM$CN$desvios <- 2


# Parametros quasi fijos
# Parametros de un LightGBM que se genera para estimar la column importance
PARAM$CN$train$clase01_valor1 <- c( "BAJA+2", "BAJA+1")
PARAM$CN$train$positivos <- c( "BAJA+2")
PARAM$CN$train$training <- c( 202101, 202102, 202103)
PARAM$CN$train$validation <- c( 202105 )
PARAM$CN$train$undersampling <- 0.1
PARAM$CN$train$gan1 <- 117000
PARAM$CN$train$gan0 <-  -3000

In [ ]:
Log("@TODO: Deberia apagar canaritos asesinos si no suma a la ganancia???! - POR AHORA NO")
# La llamada a Canaritos Asesinos
tb_importancia <- CanaritosAsesinos(
  canaritos_ratio = PARAM$CN$ratio,
  canaritos_desvios = PARAM$CN$desvios,
  canaritos_semilla = PARAM$semilla_primigenia
)


In [ ]:
# grabo la importancia, ver el archivo directamente en la carpeta
fwrite(tb_importancia,
  file="canaritos.txt",
  sep="\t"
)

In [ ]:
# verifico
ncol(dataset)
colnames(dataset)

### Variables genéticas - Feature Engineering con Algoritmo Genético

In [ ]:
# Install and load required packages
if (!require("GA")) install.packages("GA")
if (!require("dplyr")) install.packages("dplyr")
if (!require("gramEvol")) install.packages("gramEvol")
if (!require("stringr")) install.packages("stringr")
if (!require("primes")) install.packages("primes")
if (!require("parallel")) install.packages("parallel")

require("GA")
require("dplyr")
require("gramEvol")
require("stringr")
require("primes")

In [ ]:
Log("[FE - Algoritmo Genético]: Seteo paramétros")
PARAM$FE_GENALG$generations <- 100
PARAM$FE_GENALG$population <- 50
PARAM$FE_GENALG$new_variables <- 40
PARAM$FE_GENALG$max_depth <- 5

In [ ]:
is_bool_column <- function(columname) {
  tryCatch({
    if (class(max(dataset[[columname]])) %in% c('numeric', 'integer')){
      maxx = max(dataset[[columname]])
      minn = min(dataset[[columname]])
      return ((maxx - minn) == 1)
    } else {
      return (FALSE)
    }
  }, error = function(e) {
    FALSE
  })
}

get_non_bool_cols <- function(dataset) {
  all_cols <- names(dataset)
  bool_cols <- sapply(all_cols, function(col) is_bool_column(dataset, col))
  return(all_cols[!bool_cols])
}

campos_a_omitir = c('numero_de_cliente','foto_mes','azar','clase_ternaria')
campos_monetarios <- colnames(dataset)
campos_rf <- colnames(dataset)
campos_lag <- colnames(dataset)
campos_int <- colnames(dataset)

actualizar_campos = function(){
  campos_monetarios <- colnames(dataset)
  campos_rf <- colnames(dataset)
  campos_int <- colnames(dataset)
  campos_lag <- colnames(dataset)
  campos_monetarios  <<- campos_monetarios[campos_monetarios %like%
    "^(m|Visa_m|Master_m|vm_m)"]

  campos_rf <<- campos_rf[campos_rf %like%
    "^rf"]
  campos_lag <<- campos_lag[campos_lag %like%
    "(lag|delta).$"]
  campos_lag <- campos_lag[!campos_lag %in% campos_monetarios]

  campos_int <<- campos_int[!campos_int %in% c(campos_a_omitir, campos_monetarios, campos_lag, campos_rf)]
  campos_int <<- campos_int[!sapply(campos_int,is_bool_column)]
  campos_int <<- na.omit(campos_int)
}

In [ ]:
Log("[FE - Algoritmo Genético]: Actualizo campos")
actualizar_campos()

In [ ]:
safeEval <- function(expr) {
  val <- try(eval(parse(text = expr), envir = dataset), silent = TRUE)
  #if (inherits(val, "try-error") || any(is.nan(val)) || any(is.infinite(val))) {
  #  return(rep(NA, nrow(df)))
  #}
  val
}

count_operators <- function(expr) {
  text <- paste(deparse(expr), collapse = " ")  # Combine multiple lines
  str_count(text, "\\+|-|/|\\*")
}

clean_column_name_base <- function(expr_str) {
  txt <- as.character(parse(text = expr_str))
  txt <- gsub("dataset|\\[|\\]|\"", "", txt)
  txt <- gsub(",\\s*", "", txt)
  trimws(txt)
}

In [ ]:
fitness_function <- function(expr) {
  feat <- safeEval(expr)

  d <- data.frame(y = dataset$clase01, x = feat)
  colnames(d) = c('y','x')
  #d <- d[complete.cases(d), ]
  if (length(unique(d$y)) < 2) return(3)

  dtrain <- lgb.Dataset(data.matrix(d["x"]), label = d$y)

  idx <- sample(seq_len(nrow(d)), size = floor(0.2 * nrow(d)))
  dvalid <- lgb.Dataset(data.matrix(d[idx, "x", drop=FALSE]),
                        label = d$y[idx])

  valids <- list(valid = dvalid)

  model <- lgb.train(
    params = list(
      objective = "binary",
      metric = "auc",
      is_unbalance = TRUE,
      verbosity = -1,
    min_data_in_leaf = 5,
    num_leaves = 15
    ),
    data = dtrain,
    nrounds = 20,
    valids = valids,
    record = TRUE
  )

  1- tail(unlist(model$record_evals$valid$auc$eval), 1)
}

In [ ]:
grammarDef <- CreateGrammar(list(
  expr = gsrule(
      "dataset[,<m_var>]/dataset[,<m_var>]",
      "dataset[,<m_var>]/dataset[,<i_var>]",
      "dataset[,<m_var>]/dataset[,<l_var>]",
      "dataset[,<m_var>]+dataset[,<l_var>]",
      "dataset[,<m_var>]-dataset[,<l_var>]",
      "dataset[,<l_var>]/dataset[,<m_var>]",
      "dataset[,<l_var>]/dataset[,<i_var>]",
      "dataset[,<l_var>]/dataset[,<l_var>]",
      "dataset[,<l_var>]+dataset[,<l_var>]",
      "dataset[,<l_var>]-dataset[,<l_var>]",
      "dataset[,<i_var>]/dataset[,<i_var>]",
      "dataset[,<i_var>]*dataset[,<i_var>]",
      "dataset[,<rf_var>]+dataset[,<rf_var>]"
      ),
  m_var  = gvrule(setdiff(campos_monetarios,c("clase_ternaria","clase01","foto_mes",'azar'))),
  i_var  = gvrule(setdiff(campos_int,c("clase_ternaria","clase01","foto_mes",'azar'))),
  l_var  = gvrule(setdiff(campos_lag,c("clase_ternaria","clase01","foto_mes",'azar'))),
  rf_var  = gvrule(setdiff(campos_rf,c("clase_ternaria","clase01","foto_mes",'azar')))
  )
)

In [ ]:
agregarVariablesGeneticas <- function() {
  dataset[, clase01 := ifelse( clase_ternaria %in% c("BAJA+1","BAJA+2"), 1, 0 )]

  for (var in 1:PARAM$FE_GENALG$new_variables){
    actualizar_campos()
    ge_result <- GrammaticalEvolution(
      grammarDef,
      fitness_function,
      terminationCost = 0.7,
      max.depth = PARAM$FE_GENALG$max_depth,
      iterations = PARAM$FE_GENALG$generations,
      popSize = PARAM$FE_GENALG$population,
    )

    best_expr <- ge_result$best$expression
    # Generamos la columna nueva
    new_feature <- eval(parse(text = best_expr))
    coln = clean_column_name_base(parse(text=best_expr))
    colnames(new_feature) = c(coln)
    dataset <<- cbind(dataset,new_feature)
  }

  dataset[, clase01 := NULL]
}

In [ ]:
Log("[FE - Algoritmo Genético]: Agrego variables genéticas")
agregarVariablesGeneticas()
Log("[FE - Algoritmo Genético]: Variable genéticas agregadas")

### 9.7.2 Modelado

#### 9.7.2.1 Training Strategy


Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 201901, 202107 ] sin undersampling de los CONTINUA

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 201901, 202105 ]  donde se consideran el 20% de los CONTINUA

In [ ]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)

PARAM$trainingstrategy$training_pct <- 0.2


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [ ]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse(clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0)]

In [ ]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

In [ ]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if (!require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

In [ ]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

####  9.7.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Cantidad de iteraciones inteligentes de la Optimizacion Bayesiana = **10**

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en la Bayesian Optimization
  * num_leaves  [8, 256]
  * min_data_in_leaf  [8, 8192]

In [ ]:
# paquetes necesarios para la Bayesian Optimization
if(!require("DiceKriging")) install.packages("DiceKriging")
require("DiceKriging")

if(!require("mlrMBO")) install.packages("mlrMBO")
require("mlrMBO")

Definición de la Bayesian Optimization
<br> Si se desea optimizar un hiperparámetro que esta como fijo, debe QUITARSE de param_fijos y agregarse a PARAM$hipeparametertuning$hs

In [ ]:
PARAM$hipeparametertuning$num_interations <- 200
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  extra_trees = FALSE,

  max_depth = -1L, # -1 significa no limitar,  por ahora lo dejo fijo
  min_gain_to_split = 0.0, # min_gain_to_split >= 0.0
  min_sum_hessian_in_leaf = 0.001, #  min_sum_hessian_in_leaf >= 0.0
  lambda_l1 = 0.0, # lambda_l1 >= 0.0
  lambda_l2 = 0.0, # lambda_l2 >= 0.0

  bagging_fraction = 1.0, # 0.0 < bagging_fraction <= 1.0
  pos_bagging_fraction = 1.0, # 0.0 < pos_bagging_fraction <= 1.0
  neg_bagging_fraction = 1.0, # 0.0 < neg_bagging_fraction <= 1.0
  is_unbalance = FALSE, #
  scale_pos_weight = 1.0, # scale_pos_weight > 0.0

  drop_rate = 0.1, # 0.0 < neg_bagging_fraction <= 1.0
  max_drop = 50, # <=0 means no limit
  skip_drop = 0.5, # 0.0 <= skip_drop <= 1.0

  max_bin= 31,
  # num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  # early_stopping_rounds= 200
  num_iterations= 44,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 20
)


In [ ]:
PARAM$hipeparametertuning$hs <- makeParamSet(
  makeNumericParam("learning_rate", lower = 0.2, upper = 10),
  makeNumericParam("feature_fraction", lower = 0.18, upper = 0.5),
  makeNumericParam("coverage", lower = 0.05, upper = 0.90), # nuevo
  makeNumericParam("leaf_size", lower = 0.001, upper = 0.1) # nuevo
)

Función "señora caja negra"  que es llamada para verificar la realidad por la Bayesian Optimization

In [ ]:
# En x llegan los parametros de la bayesiana
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

EstimarGanancia_AUC_lightgbm <- function(x) {
  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # hago la transformacion de leaf_size y  coverage
  if(
    "leaf_size"  %in% names(param_completo)
    & "coverage"  %in% names(param_completo)
  ) {
    # primero defino el tamaño de las hojas
    param_completo$min_data_in_leaf <- pmax( 1,  round( nrow(dtrain) * param_completo$leaf_size )  )
    # luego la cantidad de hojas en funcion del valor anterior, el coverage, y la cantidad de registros
    param_completo$num_leaves <- pmin( 131072,
      pmax( 8,  round( ( param_completo$coverage * nrow( dtrain ) / param_completo$min_data_in_leaf ) ) ))
  }

  if (
    "leaf_size"  %in% names(param_completo)
    & !("coverage"  %in% names(param_completo))
  ) {
    # primero defino el tamaño de las hojas
    param_completo$min_data_in_leaf <- pmax( 1,  round( nrow(dtrain) * param_completo$leaf_size )  )
  }

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  # esta es la forma de devolver un parametro extra
  attr(AUC, "extras") <- list("num_iterations"= modelo_train$best_iter)

  # hago espacio en la memoria
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  message(format(Sys.time(), "%a %b %d %X %Y"), " AUC ", AUC)

  return(AUC)
}

seteo de la Bayesian Optimization (complejo)
<br> copiado y pegado de la documentación de la librería

In [ ]:
configureMlr(show.learner.output = FALSE)

# configuro la busqueda bayesiana,  los hiperparametros que se van a optimizar
# por favor, no desesperarse por lo complejo
obj.fun <- makeSingleObjectiveFunction(
    fn= EstimarGanancia_AUC_lightgbm, # la funcion que voy a maximizar
    minimize= FALSE, # estoy Maximizando AUC
    noisy= FALSE,
    par.set= PARAM$hipeparametertuning$hs,
    has.simple.signature= FALSE # paso los parametros en una lista
)

# cada 600 segundos guardo el resultado intermedio
ctrl <- makeMBOControl(
    save.on.disk.at.time= 600,
    save.file.path= "HT.RDATA"
)

# indico la cantidad de iteraciones que va a tener la Bayesian Optimization
ctrl <- setMBOControlTermination(
    ctrl,
    iters= PARAM$hipeparametertuning$num_interations  # cantidad de iteraciones inteligentes
)

# defino el método estandar para la creacion de los puntos iniciales
#   los "No Inteligentes"
ctrl <- setMBOControlInfill(ctrl, crit = makeMBOInfillCritEI())

# mas configuraciones
surr.km <- makeLearner(
    "regr.km",
    predict.type= "se",
    covtype= "matern3_2",
    control= list(trace = TRUE)
)

Corrida de la Bayesian Optimization,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer

Si corrío a medias y llegó a las iteraciones inteligentes, en el archivo binario HT.RDATA quedó lo ya procesado y es utilizado para retomar la corrida desde lo último que llegó a grabar.

In [ ]:
Log("[BO]: Inicio Bayesian Optimization")
if (!file.exists("HT.RDATA")) {
  bayesiana_salida <- mbo(obj.fun, learner= surr.km, control= ctrl)
} else {
  bayesiana_salida <- mboContinue("HT.RDATA") # retomo en caso que ya exista
}
Log("[BO]: Fin Bayesian Optimization")

la bayesian optimization ha corrido, extraigo los mejores hiperparametros

In [ ]:
# Almaceno los resultados de la Bayesian Optimization
#   y capturo los mejores hiperparametros encontrados

tb_bayesiana <- as.data.table(bayesiana_salida$opt.path)

# ordeno en forma descendente por AUC = y
setorder(tb_bayesiana, -y, -num_iterations)

# grabo para eventualmente poder utilizarlos en OTRA corrida
fwrite( tb_bayesiana,
  file="BO_log.txt",
  sep="\t"
)

# los mejores hiperparámetros son los que quedaron en el registro 1 de la tabla
PARAM$out$lgbm$mejores_hiperparametros <- tb_bayesiana[
  1, # el primero es el de mejor AUC
  list(learning_rate, feature_fraction, coverage, leaf_size)
]

print(PARAM$out$lgbm$mejores_hiperparametros)

### 9.7.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training</br>
Debo utilizar los mejores hiperparámetros que encontré en la optimización bayesiana

In [ ]:
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107
)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train]

# Creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow(dfinal_train) # Verifico el tamaño

##### Final Training Hyperparameters

In [ ]:
# Uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en la Bayesian Optimization
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# agrego a los hiperparametros fijos los que encontre con la Bayesian Optimization
param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)

# Hago la transformacion de leaf_size y  coverage
if (
  "leaf_size" %in% names(param_final)
  & "coverage" %in% names(param_final)
) {
  # Primero defino el tamaño de las hojas
  param_final$min_data_in_leaf <- pmax(1, round( nrow(dtrain) * param_final$leaf_size ))
  # Luego la cantidad de hojas en funcion del valor anterior, el coverage, y la cantidad de registros
  param_final$num_leaves <- pmin( 131072,
    pmax( 8,  round( ( param_final$coverage * nrow( dtrain ) / param_final$min_data_in_leaf ) ) ))
}

if (
  "leaf_size" %in% names(param_final)
  & !("coverage" %in% names(param_final))
) {
  # Primero defino el tamaño de las hojas
  param_final$min_data_in_leaf <- pmax( 1,  round( nrow(dtrain) * param_final$leaf_size )  )
}

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [ ]:
PARAM$FT$semillerio <- 300  # Cantidad de semillas

In [ ]:
if(!require("primes")) install.packages("primes")
require("primes")

In [ ]:
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
# me quedo con PARAM$semillerio  primos al azar
PARAM$FT$semillas <- sample(primos)[seq(PARAM$FT$semillerio)]

Log(paste("[Semillerio] Semillas usadas:", paste(PARAM$FT$semillas, collapse = ", ")))

In [ ]:
dir.create("modelos", showWarnings = FALSE)

i <- 0
total <- length(PARAM$FT$semillas)
for (sem in PARAM$FT$semillas) {
  nombre_arch <- paste0( "./modelos/modelo_", sem, ".txt")
  if (!file.exists(nombre_arch)) {
    param_final$seed <- sem

    set.seed(sem, kind = "L'Ecuyer-CMRG")
    final_model <- lgb.train(
      data = dfinal_train,
      param = param_final,
      verbose = -100
    )

    i <- i + 1
    Log(paste0("[Semillerio] Creando modelo ", i, "/", total, ".."))
    lgb.save(final_model, nombre_arch)

    # Grabo la importancia de variables solo 1 vez
    if (i == 1) {
      tb_importancia <- as.data.table(lgb.importance(final_model))
      archivo_importancia <- "impo.txt"

      fwrite( tb_importancia,
        file= archivo_importancia,
        sep= "\t"
      )
    }
  }
}

#### Scoring

Aplico el modelo final a los datos del futuro

In [ ]:
PARAM$trainingstrategy$future <- c(202109)

dfuture <- dataset[foto_mes %in% PARAM$trainingstrategy$future]

In [ ]:
# Aplico final_model a dfuture

tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := 0]

datos_matrix <- data.matrix(dfuture[, campos_buenos, with = FALSE])

for (isem in seq(length(PARAM$FT$semillas))) {
  sem <- PARAM$FT$semillas[ isem ]
  nombre_arch <- paste0( "./modelos/modelo_", sem, ".txt")
  final_model <- lgb.load(nombre_arch)

  prediccion <- predict(
    final_model,
    datos_matrix
  )

  tb_prediccion[, paste0("prob_", isem) := prediccion]
  tb_prediccion[, prob := prob + prediccion]

  rm(final_model)
  rm(prediccion)
  gc(full = TRUE, verbose=FALSE)
}

rm( datos_matrix)
gc(full = TRUE, verbose=FALSE)

tb_prediccion[, prob := prob/length(PARAM$FT$semillas)]

In [ ]:
# veo que hay en tb_prediccion
tb_prediccion

In [ ]:
# grabo las probabilidad del modelo
#  me va a ser util para hacer Ensembles de modelos
fwrite(tb_prediccion,
  file = "prediccion.txt",
  sep = "\t"
)

#### Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggles

In [ ]:
PARAM$kaggle$competencia <- "data-mining-analista-sr-2025-a"
PARAM$kaggle$cortes <- seq(10000, 12000, by = 200)

# Ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # Seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # Marco los primeros

  archivo_kaggle <- paste0(
    "./kaggle/KA",
    PARAM$experimento, "_",
    "s", length(PARAM$FT$semillas), "_",
    envios, ".csv"
  )

  # Grabo el csv para Kaggle
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file = archivo_kaggle,
    sep = ","
  )

  # Subida a Kaggle
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)
  mensaje <- paste0("-m 'envios=", envios, "  semilla=", PARAM$semilla_primigenia, "'")
  linea <- paste(comando, competencia, arch, mensaje)

  # Submit
  salida <- system(linea, intern=TRUE)
  Log(salida)
}

In [ ]:
if( !require("yaml")) install.packages("yaml")
require("yaml")

# grabo los parametros
write_yaml(PARAM, file="PARAM.yml")

Log("Fin del script")